# The Duality of Statistics and Deep Learning: Link Functions vs. Activation Functions
This notebook demonstrates the conceptual and mathematical equivalence between **Link Functions** in Generalized Linear Models (GLMs) and **Activation Functions** in Neural Networks.

## 1. The Core Concept
They are exactly the same concept viewed from two different mathematical perspectives. The activation function in a neural network is simply the **inverse of the link function** used in statistics.

```
Statistics (GLM):      Linear Combination <─── [Link Function (g)] ─── Expected Value (μ)
Deep Learning (NN):    Linear Combination ─── [Activation Function (f)] ───> Output (y)
```

Because they are inverses, **$f(x) = g^{-1}(x)$**.

## 2. Interactive Comparison Matrix
Every classic activation function in a neural network's output layer corresponds to a canonical link function from Generalized Linear Models (GLMs):

| Distribution | Statistical Target | Link Function ($g$) | NN Activation Function ($f = g^{-1}$) |
| :--- | :--- | :--- | :--- |
| **Gaussian / Normal** | Any continuous number | **Identity** ($x$) | **Linear / None** ($x$) |
| **Bernoulli / Binomial** | Probability between 0 and 1 | **Logit** ($\ln\frac{p}{1-p}$) | **Sigmoid** ($\frac{1}{1+e^{-x}}$) |
| **Poisson** | Counts / Rates (must be $> 0$) | **Log** ($\ln(x)$) | **Exponential** ($e^x$) |
| **Multinomial** | Multi-class probabilities | **Generalized Logit** | **Softmax** |

## 3. Mathematical Verification using Python
Let's write code to verify that the **Logit Link Function** from statistics and the **Sigmoid Activation Function** from deep learning are exact mathematical inverses of each other.

In [ ]:
import numpy as np

def logit_link(p):
    """The Link Function: Maps a probability (0, 1) to an infinite range (-inf, inf)"""
    return np.log(p / (1 - p))

def sigmoid_activation(z):
    """The Activation Function: Maps an infinite range (-inf, inf) back to a probability (0, 1)"""
    return 1 / (1 + np.exp(-z))

# --- Verification Step ---
original_probability = 0.75

# 1. Statistician maps probability to linear scale
linear_score = logit_link(original_probability)

# 2. ML Engineer maps linear scale back to probability
recovered_probability = sigmoid_activation(linear_score)

print(f"Original Probability: {original_probability}")
print(f"Logit Link Output (Linear Score): {linear_score:.4f}")
print(f"Sigmoid Activation Output (Recovered): {recovered_probability}")
print(f"Perfect Inverse Match? {np.isclose(original_probability, recovered_probability)}")

## 4. The Poisson Case: Output Layer Stability
As discussed, using the canonical Poisson activation ($e^x$) in hidden layers causes gradient explosions. Here is how you safely pair an exponential output layer with a Poisson Loss function to maintain training stability.

In [ ]:
import torch
import torch.nn as nn

class PoissonRegressionNN(nn.Module):
    def __init__(self, input_dim):
        super(PoissonRegressionNN, self).__init__()
        # Hidden layers use stable activations like ReLU
        self.hidden = nn.Linear(input_dim, 10)
        self.relu = nn.ReLU()
        # Output layer maps to continuous linear score
        self.output_layer = nn.Linear(10, 1)
        
    def forward(self, x):
        x = self.relu(self.hidden(x))
        linear_score = self.output_layer(x)
        # Canonical Poisson Link Inverse applied at the final step
        return torch.exp(linear_score)

# Example initialization
model = PoissonRegressionNN(input_dim=5)
criterion = nn.PoissonNLLLoss(log_input=False) # Pairs with raw explicit exp() output

print("Model architecture successfully built!")
print(model)